## Metadata Extraction

Extract legal metadata (case citations, dates, court names) from documents using regex.

### Import Libraries

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from pathlib import Path
import re
import json

### Define Data Paths

In [ ]:
# Define base paths - use extracted txt files from Notebook 01
BASE_DIR = Path("../data")
TXT_DIR = BASE_DIR / "txt"

### Load Extracted Text Files

In [ ]:
# Load all txt files for metadata extraction
try:
    documents = SimpleDirectoryReader(str(TXT_DIR)).load_data()
    print(f"Loaded {len(documents)} documents")
except Exception as e:
    print(f"Error loading documents: {e}")
    documents = []

Loaded 4 documents


### Define Metadata Extraction Functions

In [ ]:
# Extract Supreme Court case citations (e.g., "2025 INSC 443")
def extract_insc_citation(text):
    pattern = r'(\d{4})\s+INSC\s+(\d+)'
    matches = re.findall(pattern, text)
    return matches

# Extract SCC citations (e.g., "(2006) 1 SCC 1")
def extract_scc_citation(text):
    pattern = r'\((\d{4})\)\s+(\d+)\s+SCC\s+(\d+)'
    matches = re.findall(pattern, text)
    return matches

# Extract dates (e.g., "24 March, 2025")
def extract_dates(text):
    pattern = r'(\d{1,2}\s+\w+\s+\d{4})'
    matches = re.findall(pattern, text)
    return matches[:5]  # Limit to first 5 dates

# Extract court bench names
def extract_bench(text):
    pattern = r'Bench:\s*([A-Z][^.\n]+)'
    matches = re.findall(pattern, text)
    return matches

### Test Extraction Functions

In [ ]:
# Test extraction on first document
if documents:
    sample_text = documents[0].text
    
    print("Extracted from first document:")
    print(f"  INSC Citations: {extract_insc_citation(sample_text)}")
    print(f"  SCC Citations: {extract_scc_citation(sample_text)}")
    print(f"  Dates: {extract_dates(sample_text)}")
    print(f"  Bench: {extract_bench(sample_text)}")

Extracted from first document:
  INSC Citations: [('2025', '443')]
  SCC Citations: [('2006', '1', '1'), ('2024', '6', '461'), ('2025', '2', '641')]
  Dates: ['01 OF 2025', '21 of 2014', '81 of 2024', '01 of 2024', '08 of 2024']
  Bench: ['Vikram Nath']


### Create Chunks with SentenceSplitter

In [ ]:
# Split documents into chunks before adding metadata
if documents:
    splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"Created {len(nodes)} chunks")

Created 57 chunks


### Add Extracted Metadata to Chunks

In [ ]:
# Add metadata to each chunk
for node in nodes:
    text = node.text
    
    # Extract metadata
    insc_citations = extract_insc_citation(text)
    scc_citations = extract_scc_citation(text)
    dates = extract_dates(text)
    bench = extract_bench(text)
    
    # Add to node metadata (only if found)
    if insc_citations:
        node.metadata["insc_citations"] = str(insc_citations)
    if scc_citations:
        node.metadata["scc_citations"] = str(scc_citations)
    if dates:
        node.metadata["dates"] = str(dates)
    if bench:
        node.metadata["bench"] = str(bench)

print(f"Added metadata to {len(nodes)} chunks")

Added metadata to 57 chunks


### Explore Enriched Chunks

In [ ]:
# Print metadata of first chunk with citations
for node in nodes:
    if "insc_citations" in node.metadata or "scc_citations" in node.metadata:
        print("Chunk with citations:")
        print(json.dumps(node.metadata, indent=2))
        print(f"\nText preview: {node.text[:200]}...")
        break

Chunk with citations:
{
  "file_path": "/Users/apple/Documents/Code Playground/Lexi Bot/notebooks/../data/txt/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_name": "A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_type": "text/plain",
  "file_size": 20795,
  "creation_date": "2026-03-12",
  "last_modified_date": "2026-03-12",
  "insc_citations": "[('2025', '443')]",
  "dates": "['01 OF 2025']",
  "bench": "['Vikram Nath']"
}

Text preview: A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025



Author: Vikram ...


### Count Chunks with Metadata

In [ ]:
# Count how many chunks have each type of metadata
metadata_counts = {
    "insc_citations": 0,
    "scc_citations": 0,
    "dates": 0,
    "bench": 0
}

for node in nodes:
    for key in metadata_counts:
        if key in node.metadata:
            metadata_counts[key] += 1

print("Chunks with metadata:")
for key, count in metadata_counts.items():
    print(f"  {key}: {count} chunks")

Chunks with metadata:
  insc_citations: 4 chunks
  scc_citations: 8 chunks
  dates: 37 chunks
  bench: 4 chunks


### List All Unique Citations Found

In [ ]:
# Collect all unique citations across all chunks
all_insc = set()
all_scc = set()

for node in nodes:
    if "insc_citations" in node.metadata:
        for citation in eval(node.metadata["insc_citations"]):
            all_insc.add(f"{citation[0]} INSC {citation[1]}")
    if "scc_citations" in node.metadata:
        for citation in eval(node.metadata["scc_citations"]):
            all_scc.add(f"({citation[0]}) {citation[1]} SCC {citation[2]}")

print("Unique INSC citations:")
for citation in sorted(all_insc):
    print(f"  - {citation}")

print("\nUnique SCC citations:")
for citation in sorted(all_scc):
    print(f"  - {citation}")

Unique INSC citations:
  - 2025 INSC 443
  - 2025 INSC 447

Unique SCC citations:
  - (2002) 2 SCC 244
  - (2006) 1 SCC 1
  - (2022) 11 SCC 761
  - (2024) 3 SCC 27
  - (2024) 6 SCC 461
  - (2025) 2 SCC 641


### Summary Statistics

In [ ]:
# Print metadata extraction summary
print("=== Metadata Extraction Summary ===")
print(f"\nInput:")
print(f"  Documents: {len(documents)}")
print(f"  Total chunks: {len(nodes)}")
print(f"\nExtracted metadata:")
print(f"  Chunks with INSC citations: {metadata_counts['insc_citations']}")
print(f"  Chunks with SCC citations: {metadata_counts['scc_citations']}")
print(f"  Chunks with dates: {metadata_counts['dates']}")
print(f"  Chunks with bench info: {metadata_counts['bench']}")
print(f"\nUnique citations found:")
print(f"  INSC: {len(all_insc)}")
print(f"  SCC: {len(all_scc)}")

=== Metadata Extraction Summary ===

Input:
  Documents: 4
  Total chunks: 57

Extracted metadata:
  Chunks with INSC citations: 4
  Chunks with SCC citations: 8
  Chunks with dates: 37
  Chunks with bench info: 4

Unique citations found:
  INSC: 2
  SCC: 6
